# NOIR — Build APK di Google Colab

**Kenapa Colab bisa, HP tidak bisa:** proyek ini butuh heap Gradle **4 GB**, JDK 17, dan Android SDK 37.
Colab gratis memberi **~12–13 GB RAM** dan disk puluhan GB. Itu cukup.

**Estimasi waktu:** 30–55 menit (Gradle, NDK/CMake, dan kompilasi native CPU untuk arm64 + x86_64).

APK ini **tidak menyertakan bobot model AI**. Terjemahan lirik memakai Tencent Hy-MT2 secara lokal; saat pertama kali dipakai, pengguna bisa memilih mengunduh model sekitar **1,13 GB**. Setelahnya, inferensi luring berjalan di perangkat. Teks lirik tidak dikirim ke layanan penerjemah.

**Cara pakai:**
1. Buka [colab.research.google.com](https://colab.research.google.com)
2. **File → Upload notebook** → pilih file `.ipynb` ini
3. **Runtime → Run all**
4. Tunggu. APK akan otomatis terunduh ke perangkat kamu.

> ⚠️ Jangan pilih runtime GPU. Pakai **CPU saja** — build Android tidak butuh GPU,
> dan runtime CPU lebih stabil dan lebih gampang didapat.

---


## 1. Cek mesin

Pastikan RAM cukup sebelum buang waktu.

In [ ]:
!echo "=== RAM ===" && free -h | head -2
!echo "=== Disk ===" && df -h /content | tail -1
!echo "=== CPU ===" && nproc

## 2. Pasang JDK 17

Proyek ini pakai `jdkToolchain = 17`. JDK bawaan Colab biasanya bukan 17, jadi kita pasang dan arahkan `JAVA_HOME` ke sana.

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y openjdk-17-jdk > /dev/null
echo "JDK 17 terpasang:"
/usr/lib/jvm/java-17-openjdk-amd64/bin/java -version 2>&1 | head -2

In [ ]:
%env JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
%env PATH=/usr/lib/jvm/java-17-openjdk-amd64/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin
!java -version 2>&1 | head -1

## 3. Pasang Android SDK

Yang dibutuhkan proyek (sudah saya verifikasi tersedia di repo SDK Google):

| Kebutuhan | Paket |
|---|---|
| `compileSdk = 37` | `platforms;android-37.0` |
| `targetSdk = 36` | `platforms;android-36` |
| AGP 9.1.1 | `build-tools;37.0.0` |
| umum | `platform-tools` |
| native AI luring | `ndk;27.2.12479018` |
| CMake llama.cpp | `cmake;3.22.1` |

> Catatan: paketnya bernama `android-37.0`, **bukan** `android-37`. Foldernya pun
> `android-37.0`, dan `source.properties`-nya berisi `AndroidVersion.ApiLevel=37.0`
> dengan `IsBaseSdk=true` — itulah yang dicocokkan AGP untuk `compileSdk = 37`.
> Ini sudah saya periksa langsung dari arsip `platform-37.0_r02.zip`.

Sel ini mengunduh lebih dari 1 GB (termasuk NDK). Biarkan sampai selesai; model AI tidak diunduh di Colab dan tidak dimasukkan ke APK.

In [ ]:
%%bash
set -e

export ANDROID_HOME=/opt/android-sdk
mkdir -p "$ANDROID_HOME/cmdline-tools"
cd /tmp

# Unduh command-line tools
wget -q "https://dl.google.com/android/repository/commandlinetools-linux-13114758_latest.zip" -O clt.zip
unzip -q -o clt.zip -d "$ANDROID_HOME/cmdline-tools"

# PENTING: zip berisi "cmdline-tools/", tapi sdkmanager harus berada di
# "cmdline-tools/latest/". Tanpa langkah ini sdkmanager menolak jalan.
rm -rf "$ANDROID_HOME/cmdline-tools/latest"
mv "$ANDROID_HOME/cmdline-tools/cmdline-tools" "$ANDROID_HOME/cmdline-tools/latest"

echo "sdkmanager ada di:"
ls -l "$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager"

In [ ]:
%%bash
set -euo pipefail
export ANDROID_HOME=/opt/android-sdk
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export PATH="$JAVA_HOME/bin:$PATH"
SDKM="$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager"

# Terima semua lisensi (kalau tidak, instalasi berhenti menunggu input)
yes | "$SDKM" --licenses > /dev/null 2>&1 || true

"$SDKM" --install   "platform-tools"   "platforms;android-37.0"   "platforms;android-36"   "build-tools;37.0.0"   "ndk;27.2.12479018"   "cmake;3.22.1" 2>&1 | tail -8

echo
echo "=== platform terpasang ==="
ls "$ANDROID_HOME/platforms"
echo "=== NDK/CMake terpasang ==="
ls "$ANDROID_HOME/ndk"
ls "$ANDROID_HOME/cmake"

## 4. Ambil kode sumber

Pilih **salah satu** dari dua cara di bawah.

### Cara A — Unggah zip (paling gampang, tidak perlu GitHub)

Pakai ini kalau kamu punya file **`Noir-source.zip`** dari saya (rebrand 100% Noir + perbaikan search & UI).
Semua zip nama lain / versi lama SUDAH BASI — jangan dipakai.
Jalankan sel ini, lalu pilih file-nya lewat pemilih berkas yang muncul.


In [ ]:
# CARA A — unggah Noir-source.zip
from google.colab import files
import zipfile, os, shutil

if os.path.exists('/content/Noir'):
    shutil.rmtree('/content/Noir')

uploaded = files.upload()   # pilih Noir-source.zip (zip rebrand terbaru!)
if not uploaded:
    raise SystemExit("Tidak ada file yang diunggah.")
nama_zip = list(uploaded.keys())[0]
print("File yang diunggah:", nama_zip)
if not nama_zip.endswith(('Noir-source.zip', 'noir-source-fix19.zip')):
    raise SystemExit("❌ Pilih Noir-source.zip (atau noir-source-fix19.zip), bukan zip versi lama.")

with zipfile.ZipFile(nama_zip) as z:
    z.extractall('/content/_ekstrak')

# zip baru berisi folder "Noir/"; zip lama mungkin "AdiXtream/" — samakan jadi "Noir/"
if os.path.isdir('/content/_ekstrak/Noir'):
    src = '/content/_ekstrak/Noir'
elif os.path.isdir('/content/_ekstrak/AdiXtream'):
    src = '/content/_ekstrak/AdiXtream'
else:
    src = '/content/_ekstrak/' + os.listdir('/content/_ekstrak')[0]
shutil.move(src, '/content/Noir')
shutil.rmtree('/content/_ekstrak', ignore_errors=True)

# GUARD fix19: cek fitur lama, sumber lirik, dan integrasi AI luring.
ma = open('/content/Noir/app/src/main/java/com/lagradost/cloudstream3/MainActivity.kt').read()
fsp = open('/content/Noir/app/src/main/java/com/lagradost/cloudstream3/ui/player/FullScreenPlayer.kt').read()
sf  = open('/content/Noir/app/src/main/java/com/lagradost/cloudstream3/ui/search/SearchFragment.kt').read()
st  = open('/content/Noir/app/src/main/res/values/styles.xml').read()
fh  = open('/content/Noir/app/src/main/res/layout/fragment_home.xml').read()
au  = open('/content/Noir/app/src/main/java/com/lagradost/cloudstream3/utils/NoirAuraView.kt').read()
pl  = open('/content/Noir/app/src/main/res/layout/player_custom_layout.xml').read()
yt  = open('/content/Noir/library/src/jvmCommonMain/kotlin/com/lagradost/cloudstream3/extractors/YoutubeExtractor.jvmCommon.kt').read()
hph = open('/content/Noir/app/src/main/java/com/lagradost/cloudstream3/ui/home/HomeParentItemAdapterPreview.kt').read()
nt  = open('/content/Noir/app/src/main/java/com/lagradost/cloudstream3/utils/NoirTune.kt').read()
cs3 = open('/content/Noir/app/src/main/java/com/lagradost/cloudstream3/ui/player/CS3IPlayer.kt').read()
def _baca(jalur):
    try:
        return open(jalur).read()
    except FileNotFoundError:
        return ''
hr   = _baca('/content/Noir/app/src/main/java/com/lagradost/cloudstream3/utils/NoirHiRes.kt')
lyr  = _baca('/content/Noir/app/src/main/java/com/lagradost/cloudstream3/utils/NoirLyrics.kt')
acu  = _baca('/content/Noir/app/src/main/java/com/lagradost/cloudstream3/utils/AppContextUtils.kt')
mfp  = _baca('/content/Noir/app/src/main/AndroidManifest.xml')
prv  = _baca('/content/Noir/app/src/main/java/com/lagradost/cloudstream3/utils/NoirSpotiflacProvider.kt')
trn  = _baca('/content/Noir/app/src/main/java/com/lagradost/cloudstream3/utils/NoirOfflineLyricsTranslator.kt')
bridge = _baca('/content/Noir/app/src/main/java/com/lagradost/cloudstream3/utils/NoirLlamaBridge.kt')
native = _baca('/content/Noir/app/src/main/cpp/NoirLlamaNative.cpp')
cmake = _baca('/content/Noir/app/src/main/cpp/CMakeLists.txt')
player = _baca('/content/Noir/app/src/main/java/com/lagradost/cloudstream3/ui/music/NoirMusicPlayerActivity.kt')
gradle = _baca('/content/Noir/app/build.gradle.kts')
notices = _baca('/content/Noir/app/src/main/assets/NOIR_AI_NOTICES.txt')
strings = _baca('/content/Noir/app/src/main/res/values/strings.xml')

if ('if (lastError != null) {' not in ma
        or 'playerHostView?.callbacks?.nobarLink()' not in fsp
        or 'binding?.searchMasterRecycler?.adapter as? ParentItemAdapter' not in sf
        or st.count('@color/colorOnPrimaryNoir') < 2
        or 'noir_backdrop' not in fh
        or 'noir_hero' not in fh
        or 'NoirAuraView' not in fh
        or 'NOIR_DRIFT_FPS' not in au
        or 'noir_hero_progress' not in fh
        or 'noir_auto_moved_from_nonfilm' not in ma
        or 'chipStyle' not in st
        or '360dp' not in fh
        or 'player_subs_style_btt' not in pl
        or 'pipedFallback' not in yt
        or 'showSoftInput' not in hph
        or 'sortedByDescending { it.averageBitrate }' not in yt
        or 'NOIR fix14' not in fsp
        or 'DownloaderTestImpl.init(null)' not in ma
        or 'NoirSpotiflac.ensureLoaded' not in ma
        or 'adaptivePenalty' not in nt
        or 'onBandwidthEstimate' not in cs3
        or 'getFileUrl' not in hr
        or 'noirqz://' not in prv
        or 'noirBuildQualityChips' not in sf
        or 'timedLyricsData' not in lyr
        or 'NoirMusicPlayerActivity.openFrom' not in acu
        or '.ui.music.NoirMusicPlayerActivity' not in mfp
        or 'Hy-MT2-1.8B-Q4_K_M.gguf' not in trn
        or 'a0c709d9fac510f2c807aa3af52872340dc37a4a' not in trn
        or 'dc5f44fcf1fa496ee7ad725982c0c8c553a4de00259b53af84c4b89fb0c06699' not in trn
        or 'NoirOfflineLyricsTranslator.translate' not in lyr
        or 'translate.googleapis.com' in lyr
        or 'mymemory.translated.net' in lyr
        or 'external fun nativeGenerate' not in bridge
        or 'external fun nativeResetCancel' not in bridge
        or 'Java_com_lagradost_cloudstream3_utils_NoirLlamaBridge_nativeGenerate' not in native
        or 'Java_com_lagradost_cloudstream3_utils_NoirLlamaBridge_nativeResetCancel' not in native
        or 'refs/tags/v0.5.0' not in cmake
        or 'SHA256=fef9ed754f4e031fb5c663c29260feda4ebc241abb68d64a81c0f1df5f1748e2' not in cmake
        or 'ndkVersion = "27.2.12479018"' not in gradle
        or 'abiFilters += setOf("arm64-v8a", "x86_64")' not in gradle
        or 'downloadFirst = true' not in player
        or 'noir_music_model_download_prompt' not in player
        or 'noir_music_model_license' not in player
        or 'Apache License 2.0' not in notices
        or 'MIT' not in notices
        or 'noir_music_translation_low_memory' not in strings):
    raise SystemExit(
        "❌ ZIP SALAH / tidak lengkap — zip harus Noir-source.zip (atau noir-source-fix19.zip).\n"
        "   Hapus zip lama, unduh file terbaru, lalu jalankan sel ini lagi.")
print("✅ fix19 benar — fitur lama, terjemahan lokal, lisensi, dan runtime native terdeteksi.")
print("Kode siap di /content/Noir")
print("Jumlah file:", sum(len(f) for _,_,f in os.walk('/content/Noir')))


### Cara B — Clone dari GitHub

Pakai ini kalau kamu sudah menaruh kodenya di repo GitHub sendiri.
Ganti URL-nya dulu.

In [ ]:
# CARA B — clone dari GitHub (ganti URL-nya!)
# !git clone --depth 1 https://github.com/USERNAME-KAMU/REPO-KAMU.git /content/Noir

## 5. Siapkan konfigurasi build

Dua hal:
- `local.properties` → memberi tahu Gradle di mana SDK berada.
- `UPDATE_GITHUB_USER/REPO` → sumber fitur "cek update" di dalam aplikasi.
  Boleh dibiarkan placeholder; aplikasi tetap jalan normal, cuma fitur
  cek update-nya tidak aktif.

In [ ]:
%%bash
set -e
cd /content/Noir

cat > local.properties <<EOF
sdk.dir=/opt/android-sdk
UPDATE_GITHUB_USER=USERNAME_KAMU
UPDATE_GITHUB_REPO=REPO_KAMU
EOF

chmod +x gradlew
echo "local.properties:"
cat local.properties

## 6. Build

Ini bagian yang paling lama: **30–55 menit**. Selain Gradle, sel ini mengompilasi llama.cpp native untuk arm64-v8a dan x86_64; bobot model tidak diunduh saat build. Gradle mengunduh dirinya sendiri
(9.4.1), lalu seluruh dependensi (Media3, FFmpeg/nextlib, Coil, dll).

Kalau sel ini terasa "macat" tanpa keluaran, itu normal — Gradle sedang
mengunduh. Biarkan saja.

> Kalau muncul `Killed`, gagal mengunduh NDK/CMake, atau kehabisan memori, jalankan ulang sel ini sekali. Pastikan sel pemasangan SDK selesai dan Colab tetap online.
> Colab kadang memberi VM yang lebih kecil; **Runtime → Change runtime type**
> lalu coba lagi biasanya menyelesaikan.

In [ ]:
cd /content/Noir
set -o pipefail
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export PATH="$JAVA_HOME/bin:$PATH"
export ANDROID_HOME=/opt/android-sdk
export ANDROID_SDK_ROOT=/opt/android-sdk

# Simpan SELURUH keluaran ke build.log — jangan cuma tail,
# karena pesan error asli ada di tengah, bukan di ekor.
./gradlew assembleStableDebug --no-daemon 2>&1 | tee build.log | tail -3

echo ""
echo "═══════════════ PESAN ERROR SESUNGGUHNYA ═══════════════"
grep -n -E "^e: |FAILURE:|What went wrong|^> Task .* FAILED|Caused by:|Execution failed|A problem occurred|error:|SDK location|requires Java|Unsupported class|OutOfMemory" build.log | head -60

echo ""
echo "═══════════════ 45 BARIS DI SEKITAR ERROR PERTAMA ═══════════════"
first=$(grep -n -m1 -E "^e: |What went wrong|FAILURE:" build.log | cut -d: -f1)
if [ -n "$first" ]; then
  start=$(( first > 5 ? first - 5 : 1 ))
  sed -n "${start},$(( start + 44 ))p" build.log
else
  echo "(pola tidak dikenali — 80 baris terakhir:)"
  tail -80 build.log
fi

## 7. Ambil APK-nya

In [ ]:
import glob, os
from google.colab import files

apk = glob.glob('/content/Noir/app/build/outputs/apk/**/*.apk', recursive=True)

if not apk:
    print("APK tidak ditemukan. Scroll ke atas dan periksa error pada sel build.")
else:
    # Salin ke /content dengan nama jelas
    tujuan = '/content/Noir.apk'
    import shutil
    shutil.copy(apk[0], tujuan)
    print(f"APK   : {apk[0]}")
    print(f"Ukuran: {os.path.getsize(tujuan)/1048576:.1f} MB")
    print("\nMengunduh...")
    files.download(tujuan)

## 8. Pasang di HP

1. Pindahkan `Noir.apk` ke HP (kalau belum otomatis terunduh).
2. Buka file-nya.
3. Izinkan **"Install unknown apps"** untuk browser / file manager kamu.
4. Install.

Nama aplikasi: **Noir** · Paket: `com.noir.stream.debug`

Untuk AI terjemahan lirik luring, perangkat harus menjalankan proses Android 64-bit (arm64/x86_64) dan memiliki ruang/memori cukup. Saat pertama kali tombol terjemahan dipakai, aplikasi menawarkan unduhan model Hy-MT2 sekitar 1,13 GB. Unduh hanya jika setuju, sebaiknya lewat Wi-Fi. Model tidak dibundel ke APK; sesudah terpasang, teks lirik diterjemahkan di perangkat dan tidak dikirim ke layanan terjemahan.

> Build ini adalah **debug build**. Bisa dipakai normal untuk penggunaan sendiri.
> Yang membedakannya dari release: ukurannya sedikit lebih besar dan ada label
> `debuggable`. Untuk rilis ke orang banyak, kamu perlu keystore — tanya saja
> kalau sudah sampai tahap itu.

---

## Kalau build gagal

Salin **30 baris terakhir** keluaran sel build dan tanyakan. Yang paling sering:

| Gejala | Penyebab | Solusi |
|---|---|---|
| `Killed` / OOM | VM Colab dapat RAM kecil | Runtime → Change runtime type → coba lagi |
| `SDK location not found` | `local.properties` belum dibuat | Jalankan sel nomor 5 |
| `Failed to install the following Android SDK packages` | Lisensi belum diterima | Jalankan ulang sel nomor 3 |
| `CMake` / `NDK` tidak ditemukan | Paket native belum selesai dipasang | Ulangi sel instalasi SDK, cek `ndk/27.2.12479018` dan `cmake/3.22.1` |
| `Session timed out` | Colab idle terlalu lama | Jangan biarkan tab tidak aktif; build butuh ditunggu |
